# Simple linear regression 
## Introduction


## Fitting model to data
### Theory and definitions


### Least squares


### Calculating coefficients in Python
#### Dataset


In [ ]:
import pandas as pd

data_borkman = pd.DataFrame({
    'per_C2022_fatacids': [  # percentage of C20-22 fatty acids
        17.9, 18.3, 18.3, 18.4, 18.4, 20.2, 20.3, 21.8, 21.9, 22.1, 23.1,
        24.2, 24.4],
    'insulin_sensitivity': [  # insulin sensitivity index
        250, 220, 145, 115, 230, 200, 330, 400, 370, 260, 270, 530, 375]})

print("First row of the dataset:")
print(data_borkman.head())

#### SciPy


In [ ]:
from scipy.stats import linregress

# Load the data
X = data_borkman['per_C2022_fatacids']
y = data_borkman['insulin_sensitivity']

# Calculate the coefficients
results_scipy = linregress(X, y)
slope_scipy, intercept_scipy, *_ = results_scipy  # Extract the coefficients

print("Results from linear regression (SciPy):")
# print(results_scipy)  # Print the entire result set
print(" Intercept =", round(intercept_scipy, 3))
print(" Slope =", round(slope_scipy, 3))

#### Pingouin


In [ ]:
import pingouin as pg

model_pingouin = pg.linear_regression(
    X=X,
    y=y,
    remove_na=True,     # Deletion of missing values
    coef_only=False,    # Otherwise returns only the intercept and slopes
    as_dataframe=True,  # Otherwise returns a dictionnay of results
    alpha=0.05,         # Default
)

print("Results from linear regression (Pingouin):")
print(model_pingouin.round(3))

#### statsmodels


In [ ]:
import statsmodels.formula.api as smf
import warnings

# Suppress all UserWarnings, incl. messages related to small sample size
warnings.filterwarnings("ignore", category=UserWarning)

model_statsmodels = smf.ols(
    "insulin_sensitivity ~ per_C2022_fatacids",
    data=data_borkman)
results_statsmodels = model_statsmodels.fit()

#print(results.summary())  # Classical output of the result table
print(results_statsmodels.summary2())

In [ ]:
# Get the coefficients table
coef_table_statsmodels = results_statsmodels.summary().tables[1]

print("Coefficient table extracted from statsmodels OLS results:")
print(coef_table_statsmodels)

In [ ]:
# Print the estimated regression parameters
print("Parameters of the linear regression model (statsmodels):")
print(results_statsmodels.params.round(3))

#### NumPy


In [ ]:
import numpy as np

# Fit the model using numpy.polyfit
coefficients_numpy = np.polyfit(X, y, deg=1)
# deg=1 specifies a linear fit (degree 1)
slope_numpy, intercept_numpy = coefficients_numpy

print("Results from linear regression (NumPy):")
print(" Intercept =", round(intercept_numpy, 3))
print(" Slope =", round(slope_numpy, 3))

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(3.5, 3))

# Plot the orignal data and the calculated fitted line
plt.scatter(X, y)
plt.plot(X, slope_numpy * X + intercept_numpy, color="red", lw=2)
plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity (mg/m²/min)")
plt.title("Linear regression");

## Assessing model fit


In [ ]:
# Get the assessment table
assessment_table_statsmodels = results_statsmodels.summary().tables[0]

print("Assessment table extracted from statsmodels OLS results:")
print(assessment_table_statsmodels)

### R-squared


In [ ]:
# Calculate R-squared from SciPy
r_value_scipy = results_scipy.rvalue
r_squared_scipy = r_value_scipy**2

# Print R-squared values from different packages
print("R² (SciPy) =", round(r_squared_scipy, 4))
print("R² (statsmodels) =", round(results_statsmodels.rsquared, 4))
print(f"R² (Pingouin) =", model_pingouin.r2.iloc[1].round(4))

In [ ]:
# Define functions to estimate y and compute RSS
def estimate_y(x, b_0, b_1):
    return b_0 + b_1 * x

def compute_rss(y_estimate, y):
    return sum(np.power(y - y_estimate, 2))

# Calculate RSS and TSS
rss = compute_rss(estimate_y(X, intercept_numpy, slope_numpy), y)
tss = np.sum(np.power(y - y.mean(), 2))

# Calculate and print R-squared
print(f"TSS (manual calculation): {tss:.2f}")
print(f"RSS (manual calculation): {rss:.2f}")
print(f"R² using TSS and RSS: {1 - rss/tss:.4f}")

In [ ]:
# Print R², TSS, and RSS from statsmodels attributes
print(f"TSS (statsmodels): {results_statsmodels.centered_tss:.2f}")
print(f"RSS (statsmodels): {results_statsmodels.ssr:.2f}")

### Adjusted R-squared


In [ ]:
# Get the number of observations
n = results_statsmodels.nobs  # Number of observations
#n=len(data)  # Simpler
#p = 1  # Number of predictors in simple linear regression

# Degrees of freedom of the residuals
df_residuals = results_statsmodels.df_resid

# Calculate adjusted R²
adjusted_r_squared_manual = 1-(1-r_squared_scipy)*(n-1) / df_residuals
#adjusted_r_squared = 1 - (1 - r_squared_scipy) * (n - 1) / (n - p - 1)

# Print all calculated adjusted R²
print("Adjusted R² (manual) =", round(adjusted_r_squared_manual, 4))
print("Adjusted R² (Pingouin) =", model_pingouin.adj_r2[1].round(4))
print(
    "Adjusted R² (statsmodels) =",
    results_statsmodels.rsquared_adj.round(4))

### Mean squared error


In [ ]:
# Extract MSE from the statsmodels results object
mse = results_statsmodels.mse_resid

# Calculate MSE from RSS
mse_from_rss = rss / df_residuals

print(f"MSE (statsmodels) =", mse.round(2))
print(f"MSE (from RSS and DF) =", round(mse_from_rss, 2))
print(f"MSE (RSS/(n-2)) =", round(rss / (n - 2), 2))

### Root mean squared error


In [ ]:
# Calculate RMSE from MSE
rmse = np.sqrt(mse)

print("RMSE =", round(rmse, 3))

## Assumptions and diagnostics
### Assumptions for linear regression


### Residual analysis
#### Residual plots


In [ ]:

# Get the residuals and fitted values
residuals = results_statsmodels.resid
fitted_values = results_statsmodels.fittedvalues

plt.figure(figsize=(3.5, 3))

# Residuals vs. fitted values plot
plt.scatter(fitted_values, residuals)
plt.axhline(y=0, color='r', linestyle='--')  # Add a horizontal line at zero
# sns.residplot(x=fitted_values, y=residuals)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals plot");

#### Histograms


In [ ]:

import seaborn as sns

plt.figure(figsize=(3.5, 3))

# Histogram of residuals
sns.histplot(residuals, kde=True)
# Add a kernel density estimate for smoother visualization
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of residuals");

#### Q-Q plots


In [ ]:

from statsmodels.api import qqplot
# We could also have used SciPy or Pingouin for generating Q-Q plots

_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

# Assess whether the residuals follow a normal distribution
qqplot(residuals, fit=True, line='45', ax=ax)
plt.title("Q-Q plot of residuals");

#### Scale-location plot


In [ ]:

# Retrieve a specific type of residual that is more robust for identifying 
# outliers, more info on the link https://statsmodels.org/dev/
# generated/statsmodels.stats.outliers_influence.OLSInfluence
# .get_resid_studentized_external
model_norm_residuals = results_statsmodels.get_influence(
    ).resid_studentized_internal
model_norm_residuals_abs_sqrt = np.sqrt(np.abs(model_norm_residuals))

plt.figure(figsize=(3.5, 3))

# Create the scale-location plot
sns.regplot(
    x=fitted_values,
    y=model_norm_residuals_abs_sqrt,
    ci=None,
    lowess=True,
    line_kws={'color': 'green'},
)
plt.title("Scale-location plot")
plt.xlabel("Fitted values")
plt.ylabel(r"$\sqrt{|z_i|}$");

### Diagnostic tests
#### Statsmodels diagnostic table


In [ ]:
# Get the diagnostic table
diag_table_statsmodels = results_statsmodels.summary().tables[2]

print("Diagnostic table extracted from statsmodels OLS results:")
print(diag_table_statsmodels)

#### Breusch-Pagan


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Perform the Breusch-Pagan test
breusch_pagan_test = het_breuschpagan(
    results_statsmodels.resid,
    # Provides the design matrix (exogenous variables) used in the model
    results_statsmodels.model.exog)

# Print the test results
print("Breusch-Pagan test results:")
print(f"F-statistic =", breusch_pagan_test[-2].round(3))
print(f"P value =", breusch_pagan_test[-1].round(4))

#### Exogeneity


#### Other diagnostic tools


In [ ]:
# Get the influence summary frame from the fitted model result object
influence_summary = results_statsmodels.get_influence().summary_frame()

print("First rows of the influence summary frame:")
print(influence_summary.head())

### Consequences and remedies


## Statistical inference and uncertainty
### Standard error of the regression


In [ ]:
# Calculate S (SER) manually
ser = np.sqrt(rss / df_residuals)
#np.sqrt(results_statsmodels.mse_resid)

print(f"Standard error of the regression (SER) = {ser:.3f}")

### P values and confidence intervals


In [ ]:
# Get the coefficients table (another style)
coef_table_statsmodels_2 = results_statsmodels.summary2().tables[1]

print("Coefficient table extracted from statsmodels OLS results \
(`summary2`):")
print(coef_table_statsmodels_2.round(3))

#### T-tests for the coefficients


In [ ]:
# Extract the coefficients and standard errors from the statsmodels table
intercept_coef = results_statsmodels.params["Intercept"]
intercept_stderr = results_statsmodels.bse["Intercept"]
# standard errors are in bse
slope_coef = results_statsmodels.params["per_C2022_fatacids"]
slope_stderr = results_statsmodels.bse["per_C2022_fatacids"]

# We can also extract the standard error from SciPy regression
# intercept_stderr = results_scipy.intercept_stderr
# slope_stderr = results_scipy.stderr

# Calculate the t-statistics
intercept_tvalue = intercept_coef / intercept_stderr
slope_tvalue = slope_coef / slope_stderr

# Print the standard error values and t-statistics
print("Standard errors extracted from statsmodels:")
print(f" intercept: {intercept_stderr:.3f}")
print(f" slope: {slope_stderr:.3f}")
print("t-statistics (coefficient divided by standard error):")
print(f" intercept: {intercept_tvalue:.3f}")
print(f" slope: {slope_tvalue:.3f}")

In [ ]:
from scipy.stats import t as t_dist

# Calculate the P values (two-sided test)
intercept_pvalue = 2 * (1 - t_dist.cdf(abs(intercept_tvalue), df_residuals))
slope_pvalue = 2 * (1 - t_dist.cdf(abs(slope_tvalue), df_residuals))

# Print the P values
print(f"P value for intercept = {intercept_pvalue:.5f}")
print(f"P value for slope = {slope_pvalue:.5f}")

#### Visualizing t, critical, and P values


In [ ]:
def plot_t_distribution_with_stats(
    ax, t_value, p_value, t_crit, df_residuals, alpha, title):
    """
    Helper function that plots a t-distribution with the critical
    values, t-statistic, and P value on a given axis object.
    """

    x_t = np.linspace(-5, 5, 1000)
    hx_t = t_dist.pdf(x_t, df_residuals)

    ax.plot(x_t, hx_t, lw=2, color='black')

    # Critical values
    ax.axvline(x=-t_crit, color='orangered', linestyle='--')
    ax.axvline(
        x=t_crit, color='orangered', 
        linestyle='--', label=f"t* ({t_crit:.2f})")

    # Alpha area
    ax.fill_between(
        x_t[x_t <= -t_crit], hx_t[x_t <= -t_crit], color='tomato',
        label=f"α ({alpha})")
    ax.fill_between(
        x_t[x_t >= t_crit], hx_t[x_t >= t_crit], color='tomato')

    # t-statistic
    ax.axvline(
        x=t_value, color='limegreen', 
        linestyle='-.', label=f"t ({t_value:.2f})")

    # P value area
    ax.fill_between(
        x_t[x_t <= -abs(t_value)], hx_t[x_t <= -abs(t_value)],
        color='greenyellow', label=f"P ({p_value:.3f})")
    ax.fill_between(
        x_t[x_t >= abs(t_value)], hx_t[x_t >= abs(t_value)],
        color='greenyellow')

    ax.set_xlabel("t")
    ax.set_ylabel('Density')
    ax.set_title(f"{title}; DF={df_residuals:n}")
    ax.legend()
    ax.margins(x=0, y=0)

In [ ]:

# Significance level (α)
α = 0.05

# Calculate critical t-values (two-tailed test) common to slope and intercept
t_crit = t_dist.ppf(1 - α/2, df_residuals)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(6, 3), sharex=True, sharey=True)

# Plot for Intercept
plot_t_distribution_with_stats(
    axes[0], intercept_tvalue, intercept_pvalue, t_crit, df_residuals, α,
    f"t-distribution (intercept)")

# Plot for Slope
plot_t_distribution_with_stats(
    axes[1], slope_tvalue, slope_pvalue, t_crit, df_residuals, α,
    f"t-distribution (slope)")

plt.tight_layout();

#### Confidence interval


In [ ]:
# Calculate the confidence interval (e.g., 95% confidence)
margin_of_error_intercept = t_crit * intercept_stderr
ci_intercept = np.array([
    intercept_coef - margin_of_error_intercept,
    intercept_coef + margin_of_error_intercept])

margin_of_error_slope = t_crit * slope_stderr
ci_slope = np.array([
    slope_coef - margin_of_error_slope,
    slope_coef + margin_of_error_slope])

# Print the results
print(r"95% confidence intervals:")
print(" intercept: ", ci_intercept.round(3))
print(" slope: ", ci_slope.round(3))

### Visualizing the confidence band 


In [ ]:
from matplotlib.axes import Axes
import matplotlib.patches as mpatches

# Define the helper function that plot the confidence bands
def plot_ci_manual(t, ser, n, x, x0, y0, ax=None) -> Axes:
    """
    Plots confidence bands for a linear regression using a manual calculation.

    Args:
        t: the critical t-value for the desired confidence level.
        ser: the standard error of the regression (SER).
        n: the number of observations in the dataset.
        x: the predictor variable values.
        x0: the x-values at which to calculate and plot the CI.
        y0: the predicted y-values corresponding to x0.
        ax: optional matplotlib Axes object to plot on.
            If None, the current axes are used.

    Returns:
        The matplotlib Axes object with the confidence bands plotted.
    """

    if ax is None:
        ax = plt.gca()

    ci = t * ser * np.sqrt(
        1/n + (x0 - np.mean(x))**2 / np.sum((x - np.mean(x))**2))
    ax.fill_between(x0, y0 + ci, y0 - ci, color='salmon', zorder=0)
    
    return ax

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the data and the fitted line
plt.scatter(X, y, label="Data")
plt.plot(
    X, slope_numpy * X + intercept_numpy,
    color="yellow", lw=2,
    label="Regression line")

# Calculate and plot the CIs using the previous values for t*, S, n and X
x0 = np.linspace(np.min(X), np.max(X), 100)
y0 = slope_numpy * x0 + intercept_numpy  # Calculate predicted values for x0
plot_ci_manual(t_crit, ser, n, X, x0, y0)  # Plot CI on the current axes

# Create a custom legend handle for the confidence band
confidence_band_patch = mpatches.Patch(
    color='salmon', label='95% confidence band')

plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity")
plt.title("95% confidence band (manual)")

# Get the handles and labels from the automatically generated legend
handles, labels = plt.gca().get_legend_handles_labels()

# Add the confidence band patch to the handles list
handles.append(confidence_band_patch)

# Add the legend to the plot, including all labels
plt.legend(handles=handles);

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the data and the fitted line (statsmodels)
plt.scatter(X, y, label="Data")
plt.plot(
    X, results_statsmodels.fittedvalues,
    color="yellow",
    lw=2,
    label="Regression line")

# Extract confidence intervals from statsmodels predictions
predictions = results_statsmodels.get_prediction(X)
prediction_summary_frame = predictions.summary_frame(alpha=0.05)

# Plot the confidence intervals
plt.fill_between(
    x=X,
    y1=prediction_summary_frame['mean_ci_lower'],
    y2=prediction_summary_frame['mean_ci_upper'],
    color='salmon',
    zorder=0)

# Create a custom legend handle for the confidence band
confidence_band_patch = mpatches.Patch(
    color='salmon', label='95% confidence band')

plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity")
plt.title("95% confidence band (statsmodels)")

# Get the handles and labels from the automatically generated legend
handles, labels = plt.gca().get_legend_handles_labels()

# Add the confidence band patch to the handles list
handles.append(confidence_band_patch)

# Add the legend to the plot, including all labels
plt.legend(handles=handles);

## Bootstrapping in linear regression
### Generating bootstrap samples


In [ ]:
def draw_bs_pairs_linreg(x, y, size=1):
    """
    Perform pairs bootstrap for linear regression and return coefficients.

    This function performs pairs bootstrap resampling for linear 
    regression. It takes two arrays (x and y) representing the predictor 
    and response variables, respectively. It resamples pairs of data 
    points from x and y with replacement and fits a linear regression 
    model to each bootstrap sample.

    Args:
        x: the predictor variable values (numpy array).
        y: the response variable values (numpy array).
        size: the number of bootstrap replicates to generate (default: 1).

    Returns:
        A tuple containing two numpy arrays:
        - The first array contains the bootstrap replicates of the intercept
        - The second array contains the bootstrap replicates of the slope
    """

    # Set up array of indices to sample from
    inds = np.arange(len(x))

    # Initialize arrays to store bootstrap replicates of intercept and slope
    bs_intercept_reps = np.empty(size)
    bs_slope_reps = np.empty(size)

    # Generate replicates
    for i in range(size):
        # Resample indices with replacement
        bs_inds = np.random.choice(inds, len(inds), replace=True)

        # Get the resampled data (go by pairs)
        bs_x, bs_y = x[bs_inds], y[bs_inds]

        # Fit the linear regression model using numpy's polyfit
        bs_slope_reps[i], bs_intercept_reps[i] = np.polyfit(bs_x, bs_y, 1)

    return bs_intercept_reps, bs_slope_reps

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Set the number of replicates
B = 10_000

# Generate bootstrap replicates of the intercept and slope
bs_intercept_reps, bs_slope_reps = draw_bs_pairs_linreg(X, y, size=B)

print(
    "First six calculated bootstrap coefficients of the linear regression:")
print("Intercept (β0):", bs_intercept_reps[:6].round(2))
print("Slope (β1):", bs_slope_reps[:6].round(3))

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Set the number of replicates
B = 10_000

# Generate bootstrap replicates of the intercept and slope (more Pythonic)
bs_slope_reps, bs_intercept_reps = np.array([
    np.polyfit(X[bs_inds], y[bs_inds], 1)
    for bs_inds in np.random.choice(
        np.arange(len(X)), size=(B, len(X)), replace=True)
]).T  # Transpose to separate intercepts and slopes

print("First six calculated bootstrap coefficients (pythonic):")
print("Intercept (β0):", bs_intercept_reps[:6].round(2))
print("Slope (β1):", bs_slope_reps[:6].round(3))

### Constructing confidence intervals from bootstrap replicates


In [ ]:
# Calculate the means and 95% percentile intervals
bs_intercept_m = np.mean(bs_intercept_reps)
bs_slope_m = np.mean(bs_slope_reps)

bs_intercept_ci = np.percentile(bs_intercept_reps, [2.5, 97.5])
bs_slope_ci = np.percentile(bs_slope_reps, [2.5, 97.5])

# Print the results
print("Bootstrap estimates:")
print(" β0 = ", bs_intercept_m.round(2))
print(" β1 = ", bs_slope_m.round(2))
print("Bootstrap 95% percentile interval:")
print(" β0: ", bs_intercept_ci.round(3))
print(" β1: ", bs_slope_ci.round(3))

In [ ]:

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

# Histogram bootstrap distribution for intercept
axes[0].hist(
    bs_intercept_reps,
    density=False, bins='auto',
    color='gold',
    label=r'Bootstrap $\beta_0$')
axes[0].axvline(
    x=intercept_coef,
    color='cornflowerblue', linestyle='-', lw=2,
    label=fr'Observed $\beta_0$ ({intercept_coef:.1f})')
axes[0].axvline(
    x=bs_intercept_ci[0],
    color='red', linestyle='--',
    label=f'2.5th ({bs_intercept_ci[0]:.2f})')
axes[0].axvline(
    x=bs_intercept_ci[1],
    color='red', linestyle='--',
    label=f'97.5th ({bs_intercept_ci[1]:.2f})')
axes[0].set_xlabel(r"$\beta_0$")
axes[0].set_ylabel('Frequency')
axes[0].set_title(r"Bootstrap $\beta_0$")
axes[0].legend(fontsize=9)

# Histogram bootstrap distribution for slope
axes[1].hist(
    bs_slope_reps,
    density=False, bins='auto',
    color='gold',
    label=r'Bootstrap $\beta_1$')
axes[1].axvline(
    x=slope_coef,
    color='cornflowerblue', linestyle='-', lw=2,
    label=fr'Observed $\beta_1$ ({slope_coef:.1f})')
axes[1].axvline(
    x=bs_slope_ci[0],
    color='red', linestyle='--',
    label=f'2.5th ({bs_slope_ci[0]:.2f})')
axes[1].axvline(
    x=bs_slope_ci[1],
    color='red', linestyle='--',
    label=f'97.5th ({bs_slope_ci[1]:.2f})')

axes[1].set_xlabel(r"$\beta_1$")
axes[1].set_ylabel('Frequency')
axes[1].set_title(r"Bootstrap $\beta_1$")
axes[1].legend(fontsize=9)

plt.tight_layout();

In [ ]:
from scipy.stats import norm

# Calculate the standard error for each coefficient
bs_intercept_s = np.std(bs_intercept_reps, ddof=1)
bs_slope_s = np.std(bs_slope_reps, ddof=1)

# Calculate the 95% normal margin of error
z_crit = norm.ppf((1 + (1-α))/2)
bs_intercept_w =  z_crit * bs_intercept_s
bs_slope_w =  z_crit * bs_slope_s

# Calculate normal confidence interval
bs_intercept_ci_normal = np.array([
    bs_intercept_m - bs_intercept_w, bs_intercept_m + bs_intercept_w])
bs_slope_ci_normal = np.array([
    bs_slope_m - bs_slope_w, bs_slope_m + bs_slope_w])

print("Bootstrap standard errors:")
print(f" s(β0) = {bs_intercept_s:.4f}")
print(f" s(β1) = {bs_slope_s:.4f}")
print(r"95% asymptotic CI of bootstrap distributions:")
print(" β0: ", bs_intercept_ci_normal.round(3))
print(" β1: ", bs_slope_ci_normal.round(3))

### Bootstrapping and the confidence band


In [ ]:

plt.figure(figsize=(3.5, 3))

# Visualize the variability of the regression line
for i in range(200):
    plt.plot(
        X, bs_slope_reps[i] * X + bs_intercept_reps[i],
        'y-', lw=1, alpha=0.25,
        label='Bootstrap lines' if i==0 else None)
        # Print label only once in legend

plt.plot(X, y, 'ko', linestyle='None', label='Data')
plt.plot(
    X, results_statsmodels.fittedvalues,
    color="crimson", lw=2,
    label="True regression line")
plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity")
plt.title("Bootstrap regression lines")
plt.legend();

In [ ]:

plt.figure(figsize=(3.5, 3))

# Visualize the variability of the regression line (Seaborn)
sns.regplot(
    x=X, y=y,
    ci=95, n_boot=1000, seed=111,
    scatter_kws={'color': 'k'},
    line_kws={'color': 'forestgreen', 'lw': 2})
plt.xlabel('%C20-22 fatty acids')
plt.ylabel('Insulin sensitivity')
plt.title("Bootstrap regression (Seaborn)");

### Permutation test for the slope coefficient


In [ ]:
def permute_for_linreg(X, y):
    """
    Generates a permuted sample for linear regression 
    under the null hypothesis of no relationship.

    Args:
      X: the predictor variable array.
      y: the response variable array.

    Returns:
      A new array with the values of x permuted randomly,
      and the original y array.
    """
    # Permute the values of X
    permuted_X = np.random.permutation(X)

    return permuted_X, y

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate 9999 permutation replicates of the t-statistic for β1
B = 9999  # 99 rule
perm_slope_t = np.empty(B)

# Generate permuted coefficients
for i in range(B):
    # Permute the data
    X_perm, y = permute_for_linreg(X, y)

    # Fit the model to the permuted data and extract coefficients
    # and their SE
    res = linregress(X_perm, y)
    perm_slope_t[i] = res.slope / res.stderr

# Same using super pythonic approaches
#
# perm_slope_t = np.array([
#     pg.linear_regression(np.random.permutation(X),y).iloc[-1, 3]
#     for _ in range(B)])
#
# _, perm_slope_t = map(np.array, zip(*[
#     pg.linear_regression(np.random.permutation(X), y).iloc[[0, 1], 3]
#     for _ in range(B)]))

print("First ten calculated permuted t-statistics for β1:")
print(perm_slope_t[:10].round(2))

In [ ]:
# Calculate the P value using the distribution of bootstrap t-statistic 
# for β1 in permuted pairs considering the direction of the observed 
# t-statistic slope_tvalue = results_statsmodels.tvalues.iloc[-1]
# 1. Calculate lower and upper tail
if slope_tvalue >= 0:
    p_value_slope_t_perm_up = np.mean(perm_slope_t >= slope_tvalue)
    p_value_slope_t_perm_lo = np.mean(perm_slope_t <= -slope_tvalue)
else:
    p_value_slope_t_perm_lo = np.mean(perm_slope_t <= slope_tvalue)
    p_value_slope_t_perm_up = np.mean(perm_slope_t >= -slope_tvalue)

# 2. Attribute the one-tailed P value
p_value_slope_t_perm_1t = p_value_slope_t_perm_up if slope_tvalue >= 0 \
else p_value_slope_t_perm_lo

print("One-sided t-test using permutation:")
print(f"P values for β1 = {p_value_slope_t_perm_1t:.5f}")

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the permuted β1 t-statistics
hist, bins, patches = plt.hist(
    perm_slope_t,
    density=False, bins=int(B**.5),
    color='gold',)

# Annotate the observed t-statistic for β1
plt.axvline(
    x=slope_tvalue,
    color='limegreen', linestyle='-.', lw=2,
    label=r"Observed $t_1$ "f"({slope_tvalue:.3f})")

# Determine the direction of observed β1 t-statistic and plot accordingly
if r_squared_scipy >= 0:
    # Plot the histogram of β1 t-statistics >= observed β1 t-statistic
    extreme_diffs = perm_slope_t[perm_slope_t >= slope_tvalue]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_diffs):
            patches[i].set_facecolor('lime')
else:
    # Plot the histogram of β1 t-statistics <= observed β1 t-statistic
    extreme_diffs = perm_slope_t[perm_slope_t <= slope_tvalue]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_diffs):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel(r'$t_1$')
plt.ylabel('Frequency')
plt.title(r"Distribution of $t_1$ under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label=r'Permuted $t_1$')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=fr'One-tailed P ({p_value_slope_t_perm_1t:.4f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

In [ ]:
# Maximum one-tail
p_value_permut_2t_max = 2 * max(
    p_value_slope_t_perm_lo, p_value_slope_t_perm_up)

# Sum of the tails
p_value_permut_2t_sum = p_value_slope_t_perm_lo + p_value_slope_t_perm_up

print("Permutation t-statistic for β1:")
print(f"Two-tailed P value (conservative) = {p_value_permut_2t_max:.5f}")
print(f"Two-tailed P value (both tails) = {p_value_permut_2t_sum:.5f}")

## Prediction
### Making predictions
### Prediction with Python


In [ ]:
from sklearn.linear_model import LinearRegression

# Create a LinearRegression object
model_sklearn = LinearRegression()

# scikit-learn requires a specific data format, think it uses vectors 
# and matrices, so either extract DataFrame, transform Series to DataFrame
X_matrix = data_borkman[['per_C2022_fatacids']]  # same as X.to_frame()
# or reshape X Series values to a 2D NumPy array
# X_matrix = X.values.reshape(-1, 1)

# Fit the model to the data
model_sklearn.fit(X_matrix, y)

# Extract the coefficients
intercept_sklearn = model_sklearn.intercept_
slope_sklearn = model_sklearn.coef_[0]

# Print the coefficients
print("Intercept (sklearn) =", intercept_sklearn.round(3))
print("Slope (sklearn) =", slope_sklearn.round(3))

In [ ]:

x_line = np.array([X.min(), X.max()])

# Predict y-values using the fitted model
y_line = model_sklearn.predict(x_line.reshape(-1, 1))  # Reshape for sklearn

plt.figure(figsize=(3.5, 3))

# Plot the data and the fitted line
plt.scatter(X, y)
plt.plot(x_line, y_line, color="crimson", lw=3)
plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity")
plt.title("Linear regression (sklearn)");

In [ ]:
# Create new/unseen data points for prediction
X_new = np.array([20, 24])
X_new_matrix = X_new.reshape(-1, 1)  # Reshape for sklearn

# Predict the response variable for the new data points
y_pred_new = model_sklearn.predict(X_new_matrix)

print(f"Predicted insulin sensitivity for:")
# Print the predictions
for x_i, y_i in zip(X_new_matrix, y_pred_new):
    print(f"  {x_i[0]:.0f}% C20-22 fatty acids = {y_i:.2f} mg/m²/min")

### Model performance


In [ ]:
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error)

# Predict y-values for all initial x-vlaues using the fitted model
y_pred = model_sklearn.predict(X_matrix.values)

# Calculate evaluation metrics
mse_sklearn  = mean_squared_error(y, y_pred)
rmse_sklearn  = np.sqrt(mse)
r2_sklearn = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print("Model evaluation metrics (sklearn):")
print(f"MSE = {mse_sklearn :.2f}")
print(f"RMSE = {rmse_sklearn :.4f}")
print(f"R² = {r2_sklearn :.6f}")
print(f"MAE = {mae :.2f}")

In [ ]:
# Calculate and print RSS/n
rss_over_n = rss / n  # results_statsmodels.nobs
print("RSS/n = ", round(rss_over_n, 2))

### Prediction interval 


In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the data and the fitted line derived from sklearn
plt.scatter(X, y, label="Data")
plt.plot(
    x_line, y_line, color="yellow", lw=2, label="Regression line")

# Calculate and plot the Cis using the previous values for t*, S, n and X
x0 = np.linspace(X.min(), X.max(), 100)
y0 = model_sklearn.predict(x0.reshape(-1, 1))  # Predict values for x0
plot_ci_manual(t_crit, ser, n, X, x0, y0)  # Plot CI on the current axes

# Create a custom legend handle for the confidence band
confidence_band_patch = mpatches.Patch(
    color='salmon', label="95% confidence band")

# Calculate and plot prediction intervals
pred_interval = t_crit * ser * np.sqrt(
    1 + 1/n + (x0 - X.mean())**2 / np.sum((X - X.mean())**2))

plt.plot(
    x0, y0 - pred_interval,
    "--", color="forestgreen", lw=2,
    label="95% prediction limits")
plt.plot(x0, y0 + pred_interval, "--", color="forestgreen", lw=2)

plt.xlabel("%C20-22 fatty acids")
plt.ylabel("Insulin sensitivity")
plt.title("Prediction limits (manual)")

# Get the handles and labels from the automatically generated legend
handles, labels = plt.gca().get_legend_handles_labels()

# Add the confidence band patch to the handles list
handles.append(confidence_band_patch)

# Add the legend to the plot, including all labels
plt.legend(handles=handles);

In [ ]:

_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

# Plot the data and the fitted line (statsmodels)
ax.scatter(X, y, label="Data")
ax.plot(
    X, results_statsmodels.fittedvalues,
    color="yellow", lw=3,
    label="Regression line")

# Extract confidence intervals from
predictions = results_statsmodels.get_prediction(X)
prediction_summary_frame = predictions.summary_frame(alpha=0.05)

# Plot the confidence intervals
ax.fill_between(
    X,
    prediction_summary_frame['mean_ci_lower'],
    prediction_summary_frame['mean_ci_upper'],
    color='salmon', zorder=0)

# Create a custom legend handle for the confidence band
confidence_band_patch = mpatches.Patch(
    color='salmon', label="95% confidence band")

# Calculate and plot prediction intervals
ax.plot(
    X, prediction_summary_frame['obs_ci_lower'],
    linestyle="--", color="forestgreen", lw=2,
    label="95% prediction limits")
ax.plot(
    X, prediction_summary_frame['obs_ci_upper'],
    "--", color="forestgreen", lw=2)

ax.set(
    xlabel="%C20-22 fatty acids",
    ylabel="Insulin sensitivity",
    title="Prediction limits (statsmodels)")

# Get the handles and labels from the automatically generated legend
handles, labels = plt.gca().get_legend_handles_labels()

# Add the confidence band patch to the handles list
handles.append(confidence_band_patch)

# Add the legend to the plot, including all labels
ax.legend(handles=handles);

In [ ]:
# Print the prediction intervals for each x-value
for i, x_i in enumerate(X_new):
    # Calculate the prediction interval margin for the current x-value (x_i)
    pi_w = t_crit * ser * np.sqrt(
        1 + 1/n + (x_i - X.mean())**2 / np.sum((X - X.mean())**2))

    # Extract the predicted y-value for the current x-value from sklearn
    y_pred_new_i = y_pred_new[i]

    # Calculate the prediction interval
    pred_interval = np.array([
        y_pred_new_i - pi_w, y_pred_new_i + pi_w])

    # Print the prediction interval information
    print(f"Prediction for {x_i:n}% C20-22 fatty acids:")
    print(f"  Predicted value: {y_pred_new_i:.2f} mg/m²/min")
    print(f"  Prediction interval: ", pred_interval.round(4))

In [ ]:
# Calculate prediction intervals and extract relevant values
predictions = results_statsmodels.get_prediction(
    exog=dict(per_C2022_fatacids=X_new))
pred_intervals = predictions.summary_frame(alpha=0.05)[
    ['mean', 'obs_ci_lower', 'obs_ci_upper']].copy()
pred_intervals.index=X_new
pred_intervals.index.name='per_C2022_fatacids'
pred_intervals = pred_intervals.rename(
    columns={
        "mean": "y_pred", 
        "obs_ci_lower": "pi_lo", 
        "obs_ci_upper": "pi_up"})
print("Prediction intervals (statsmodels):")
print(pred_intervals.round(3))

## Advanced techniques
### Maximum likelihood estimation
#### How MLE works
#### MLE and linear regression


#### Visualization of MLE


In [ ]:

from matplotlib import collections as mc

# Define different slopes based on the coefficient obtained with OLS
slopes = [slope_sklearn, slope_sklearn * 0.8, slope_sklearn * 1.35]

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(6, 2.5), sharex=True)

# Estimate the standard deviation of the residuals
# Calculate the residuals from the scikit-learn model and its predictions
residuals = y - y_pred
resid_std = residuals.std()

# Optimize the x-range based on the OLS residuals
x_min = -5 * resid_std
x_max = 5 * resid_std
x_range = np.linspace(x_min, x_max, 1000)

# Iterate through different slopes
for i, slope in enumerate(slopes):
    # Fit the model with the current slope
    model = LinearRegression()
    model.intercept_ = intercept_sklearn  # Keep the intercept constant
    model.coef_ = np.array([slope])  # Set the new slope
    y_pred_i = model.predict(X_matrix)
    residuals_mle = y - y_pred_i

    ax = axes[i]
    # Plot predicted values as crosses at the bottom
    ax.plot(residuals_mle, [0] * len(residuals_mle), 'b+')
    # Plot the normal distribution
    ax.plot(x_range, norm.pdf(x_range, loc=0, scale=resid_std), 'k')

    # Calculate predicted likelihoods
    predicted_likelihoods = norm.pdf(residuals_mle, loc=0, scale=resid_std)
    predicted_coords = [
        [v, p] for v, p in zip(residuals_mle, predicted_likelihoods)]

    # Plot segments
    segments = [
        [[v,0], [v,p]] for v,p in zip(residuals_mle,predicted_likelihoods)]
    lc = mc.LineCollection(
        segments,
        colors='red',
        linewidths=1,
        label='Predicted likelihood')
    ax.add_collection(lc)

    # Calculate and display log-likelihood
    log_likelihood = np.sum(np.log(predicted_likelihoods))
    ax.set_title(
        "β1: "f"{slope:.1f}; LL={log_likelihood:.1f}",
        fontsize=10)
    ax.set_xlabel('Residuals insulin sensit')
    ax.set_ylabel('Likelihood')
    ax.set_yticks([])
    ax.margins(x=0, y=0.05)

plt.tight_layout();

In [ ]:
# Print the log-likelihood from statsmodels
print("Log-likelihood (statsmodels) =", round(results_statsmodels.llf, 3))

# Print the log-likelihood from the previous visualization
print(f"Log-likelihood (MLE) = {np.sum(
    np.log(norm.pdf(residuals, loc=0, scale=resid_std))):.3f}")

### Generalized linear models


## Conclusion
